# Member C - Ablation V3

Attach `asl-signs` và Shared V3; bật GPU rồi Run All. Source nằm trong Shared, không cần checkout branch. Các results V2 được lưu riêng; V3 train từ đầu. Sau mỗi config, tải Results ZIP hoặc Save Version giữ Output.


In [ ]:
MEMBER = "C"
MODE = "train"  # train / report / aggregate / test (A)
SHARED_INPUT = ""  # tự tìm Shared V3 ZIP hoặc thư mục đã giải nén
RAW_INPUT = "/kaggle/input/competitions/asl-signs"
RESTORE_INPUT = ""  # ZIP results V3 của chính member để tiếp tục configs còn thiếu
CROSS_PLAN_INPUT = ""  # B: cross_plan.zip từ A, sau vòng đầu
RESULT_INPUTS = []  # A: paths tới ZIP results B/C khi MODE="aggregate"
RUN_IDS = None  # None chạy cả queue; hoặc list run IDs
RETRY_FAILED = False


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile

EXPECTED_FREEZE_ID = "b4fd56f03a57"
SEARCH_ROOT = Path("/kaggle/input")
if not SHARED_INPUT:
    candidates = list(SEARCH_ROOT.rglob("ablation_v3_shared_" + EXPECTED_FREEZE_ID + ".zip"))
    candidates += [p.parent for p in SEARCH_ROOT.rglob("freeze.json")
                   if json.loads(p.read_text()).get("freeze_id") == EXPECTED_FREEZE_ID]
    if len(candidates) != 1:
        raise ValueError(f"Tìm thấy {len(candidates)} Shared V3 inputs. Đặt SHARED_INPUT vào ZIP hoặc thư mục có freeze.json.")
    SHARED_INPUT = str(candidates[0])
source = Path(SHARED_INPUT)
if source.is_file():
    with zipfile.ZipFile(source) as archive:
        if len(archive.namelist()) != len(set(archive.namelist())) or archive.testzip():
            raise ValueError("Shared ZIP không hợp lệ.")
        payloads = {name: archive.read(name) for name in archive.namelist() if not name.endswith("/")}
else:
    payloads = {p.relative_to(source).as_posix(): p.read_bytes() for p in source.rglob("*") if p.is_file()}
frozen = json.loads(payloads["freeze.json"])
actual = {name: hashlib.sha256(payloads[name]).hexdigest() for name in frozen["files"]}
identity = hashlib.sha256(json.dumps(actual, sort_keys=True).encode()).hexdigest()[:12]
if actual != frozen["files"] or identity != EXPECTED_FREEZE_ID or frozen["freeze_id"] != identity:
    raise ValueError("Shared khác freeze của notebook này.")
manifest = json.loads(payloads["source_manifest.json"])
for name, checksum in manifest.items():
    if hashlib.sha256(payloads["code/" + name].replace(b"\r\n", b"\n")).hexdigest() != checksum:
        raise ValueError(f"Source hash lệch: {name}")
DEPLOY_ROOT = Path("/kaggle/working") / ("islr_ablation_v3_" + identity)
for name, content in payloads.items():
    target = (DEPLOY_ROOT / name).resolve()
    if not target.is_relative_to(DEPLOY_ROOT.resolve()) or "\\" in name or ":" in name:
        raise ValueError("Unsafe Shared path.")
    if target.exists() and target.read_bytes() != content:
        raise ValueError(f"Runtime source/input bị sửa: {name}")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
CODE_ROOT = DEPLOY_ROOT / "code"
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT / "reports/ablation-study-v3/tools"))
print("Đã xác minh source, config, matrix, split:", identity)


In [ ]:
import subprocess, importlib.util
required = {"yaml": "pyyaml", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "pandas": "pandas", "tqdm": "tqdm"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import torch
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available())
if MODE == "train" and not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Kaggle Settings trước khi training.")


In [ ]:
from session_v3 import MemberSession
import pandas as pd
session = MemberSession(MEMBER, DEPLOY_ROOT, Path(RAW_INPUT), Path("/kaggle/working"))
if RESTORE_INPUT:
    session.restore_results(Path(RESTORE_INPUT))
if CROSS_PLAN_INPUT:
    session.load_cross_plan(Path(CROSS_PLAN_INPUT))
if MODE == "train":
    session.check_data()
display(pd.DataFrame(session.report()["queue"]))


In [ ]:
if MODE == "train":
    status = session.run_queue(RUN_IDS, retry_failed=RETRY_FAILED)
    display(pd.DataFrame(status["queue"]))
    print("Hoàn tất:", status["completed"], "/", status["expected"])
elif MODE in ("aggregate", "test"):
    result = session.aggregate([Path(p) for p in RESULT_INPUTS])
    print(result["status"], "winner:", result["winner"], "missing:", result["missing"])
    if MODE == "test":
        print(session.finalize_test(result))
elif MODE == "report":
    display(pd.DataFrame(session.report()["queue"]))
else:
    raise ValueError("MODE phải là train/report/aggregate/test.")


In [ ]:
results_zip = session.export_results()
from IPython.display import FileLink, display
display(FileLink(str(results_zip)))
if (session.root / "cross_plan.zip").is_file():
    display(FileLink(str(session.root / "cross_plan.zip")))


## Nghiệm thu

Một config hoàn tất phải đủ 3 seed reports, predictions đúng cohort, checkpoints, history, metrics và stage logs. ZIP được tạo sau mỗi config; Kaggle reset sẽ mất Output chưa tải/Save Version. B giữ ZIP vòng đầu để restore khi chạy cross-plan. A tổng hợp đủ B/C, gửi cross-plan cho B nếu cần, rồi nghiệm thu lần cuối. Test giữ kín đến khi khóa winner.
